#### Importing Standard Libraries

In [0]:
from pyspark.sql.functions import lit, col, lower, isnull, trim, md5, concat_ws, when, max, rank, dense_rank
from pyspark.sql.window import Window
from datetime import datetime, date
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from delta.tables import DeltaTable
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import json

#### logger notebook execution
This notebook created a logger object which captures
1. logger.info()
2. logger.exceptions()

In [0]:
%run ../utils/loggingNotebook

#### Params notebook execution 
This notebook helps in getting the following 
1. ruleCategoryList
2. ruleNameList
3. ruleMasterPath
4. ruleObjectMappingPath
5. dqStage
6. filterColumnList
7. dataQualityTable
8. jobControlTable

In [0]:
%run ../utils/params

#### Utils notebook execution
This notebook helps in getting dqFunctions definition
1. sendEmailAlertDQ
2. masterRules function
3. countCheck function
4. dqMd5Check function
5. customQueryCheck function

In [0]:
%run ../utils/utilsMethods

#### Reading data from widgets

In [0]:
sourceObjectName = dbutils.widgets.get('sourceObjectName').strip().lower()
sourceSystem = dbutils.widgets.get('sourceSystem').strip().lower()

#### Active Flag Check

In [0]:
dfConfig = readSparkCsv(jobConfigPath)
dfActiveSource = dfConfig.select("*").filter((lower(trim(col("activeFlag")))=='y') & (lower(trim(col("sourceSystem"))) == sourceSystem) & (lower(trim(col("sourceObjectName"))) == sourceObjectName))
if dfActiveSource.isEmpty():
  dbutils.notebook.exit("Notebook exited as active flag is N")
else:
  pass

In [0]:
auditAdbPipelineID = spark.sql(f"""select adbPipelineID from {pipelineAuditTable} where sourceObjectName = "{sourceObjectName}" and sourceSystem = "{sourceSystem}" and stageOrder = {curatedOrderStage} order by adbPipelineID desc limit 1""").collect()[0][0]

In [0]:
#Fetching the path of the notebook
notebookPath = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebookName = notebookPath.split("/")[-1] #Splitting the path to get notebook name

if sourceObjectName != "" and sourceSystem != "":
    if "." in sourceObjectName:
        schema = sourceObjectName.split('.')[0]
        objectName = sourceObjectName.split('.')[1]
        adbpipelineID = auditAdbPipelineID.replace(curatedStage,dqStage)
        properties={'custom_dimensions': {'adbPipelineID':f'{adbpipelineID}' \
                                          ,'auditAdbPipelineID': f'{auditAdbPipelineID}' \
                                          ,'notebookName':f'{notebookName}','sourceName':f'{sourceSystem}','tableName':f'{sourceObjectName}'}}
    else:
        schema = None
        objectName = sourceObjectName
        adbpipelineID = auditAdbPipelineID.replace(curatedStage,dqStage)
        properties={'custom_dimensions':{'adbPipelineID':f'{adbpipelineID}' \
                                         ,'auditAdbPipelineID': f'{auditAdbPipelineID}'\
                                         ,'notebookName':f'{notebookName}','sourceName':f'{sourceSystem}','tableName':f'{sourceObjectName}'}}
else:  
    logger.exception("Empty sourceObjectName or sourceSystem", extra=properties)
    raise Exception("Following field(s) is empty - sourceObjectName, sourceSystem.")

#### Reading from jobConfigPath and jobControlTable from Path
jobConfig path will give
1. dateColumnName
2. sourceObjectName
3. sourceSystem 

jobControlTable will give
1. lastRunDate
2. currentRunDate

In [0]:
try:
    logger.info("Reading data from config file",extra=properties)
    dfConfig = dfConfig.select([when(lower(trim(col(c)))=="null",None).otherwise(col(c)).alias(c) for c in dfConfig.columns])
    
    # Filtering data on sourceObjectName and sourceSystem
    config = dfConfig.where((trim(lower(col("sourceObjectName")))== sourceObjectName.lower()) & (trim(lower(col("sourceSystem")))== sourceSystem.lower())).collect()[0].asDict()
    logger.info("Configuration file read and entries are validated", extra = properties)
except Exception as e:
    logger.exception(f"No configuration data found for {sourceObjectName} and {sourceSystem}. Error message: {e}", extra=properties)
    raise e

In [0]:
try:
    dfJobControl = spark.sql(f"SELECT * FROM {jobControlTable}")
    jobControl = dfJobControl.where((trim(lower(col("sourceSystem"))) == sourceSystem.lower())).collect()[0].asDict()
    logger.info("jobControl table read successfully", extra = properties)
except Exception as e:
    logger.exception(f"No data found in jobControl table for {sourceObjectName} and {sourceSystem}. Error message: {e}", extra=properties)
    raise e

#### Reading variables and paths from config file and jobcontrol table

In [0]:
try:
    logger.info("Creating variables based on configuration file and jobControl table ", extra=properties)  
    lastRunDate = jobControl['lastRunDate']
    currentRunDate = jobControl['currentRunDate']
    deltaPath = baseAdlsPath+config['curatedPath']
    primaryKey = config['pkColumnName']
    dateColumnName = config['dateColumnName']
    activeFlag = config['activeFlag']
except Exception as e:
    logger.exception(f"Missing required values in configuration file. Error message: {e}", extra=properties)
    raise e

#### rulemaster validation

##### Validate if csv file is present in ruleMaster directory & it is not empty

In [0]:
# Reading the rulemaster csv file from the ADLS location, into a dataframe
try:
    logger.info(f"Checking if ruleMaster file exists at the specified path '{ruleMasterPath}' ", extra = properties)
    if not dbutils.fs.ls(ruleMasterPath):
        logger.exception(f"rulemaster file not be found at the specified path '{ruleMasterPath}'. Please verify the path.", extra = properties)
        raise FileNotFoundError("rulemaster file not found at the specified path. Please verify the path")
    else:
        logger.info(f"ruleMaster file is present at the specified path '{ruleMasterPath}'", extra = properties)
        dfRuleMaster = readSparkCsv(ruleMasterPath+ '/*.csv')
        # to check if dfRuleMaster dataframe is empty or not
        if dfRuleMaster.isEmpty():
            logger.exception("no entries found in ruleMaster file", extra = properties)
            raise ValueError("no entries found in ruleMaster file")
        else:
            logger.info("ruleMaster file read successfully", extra = properties)
            
except Exception as e:
    logger.exception(f"ruleMaster file not found at the specified path'{ruleMasterPath}'. Please verify the path : {e}", extra=properties)
    raise e

#### ruleObjectMapping validation

##### Validate if csv files present in ruleObjectMapping directory & it is not empty

In [0]:
# Reading the ruleObjectMapping file from the ADLS location, into a dataframe
try:
    logger.info(f"Checking if ruleObjectMapping file exists at the specified path '{ruleObjectMappingPath}' ", extra = properties)
    if not dbutils.fs.ls(ruleObjectMappingPath):
        logger.exception(f"ruleObjectMapping file not found at the specified path {ruleObjectMappingPath}. Please check the path.", extra = properties)
        raise FileNotFoundError("ruleObjectMapping file not found at the specified path. Please check the path")
    else:
        logger.info(f"ruleObjectMapping file exists at the path '{ruleObjectMappingPath}'", extra = properties)
        dfRuleObjectMapping = readSparkCsv(ruleObjectMappingPath+ '/*.csv')
        if dfRuleObjectMapping.isEmpty():
            logger.exception("No entries found for ruleObjectMapping file", extra = properties)
            raise ValueError("No entries found for ruleObjectMapping file")
        else:
            logger.info("ruleObjectMapping file read successfully", extra = properties)
            
except Exception as e:
    logger.exception(f" ruleObjectMapping file not found at the specified path at {ruleObjectMappingPath}. Please verify the path: {e}", extra=properties)
    raise e

#### Validating ruleID, ruleName, ruleCategory and IsActive

In [0]:
try:
    if dfRuleMaster.filter(lower(col('isActive')) == 'y'):
        dfActive = dfRuleMaster.filter(lower(col('isActive')) == 'y')
        if dfActive.filter(col("ruleID").isNull()).count() == 0:
            logger.info("ruleID has no null entries")
            if dfActive.select("ruleID").distinct().count() == dfActive.count():
                logger.info("ruleID has unique entries", extra = properties)
            else:
                logger.exception("ruleID has duplicate entries", extra = properties)
        else:
            logger.exception("ruleID has null entries")
            raise ValueError("ruleID has null entries")
    
        # ruleName should not be null & unique and within defined categories
        if dfActive.filter(col("ruleName").isNull()).count() == 0:
            logger.info("ruleName has no null values", extra = properties)
            if dfActive.select("ruleName").distinct().count() == dfActive.count():
                logger.info("ruleName has unique entries", extra = properties)
                if dfActive.filter(col('ruleName').isin(ruleNameList)):
                    logger.info("ruleName is within defined ruleName list", extra = properties)
                else:
                    logger.exception("ruleName is not within defined ruleName list", extra = properties)
            else:
                logger.exception("ruleName has duplicate entries", extra = properties)
        else:
            logger.exception("ruleName has null entries", extra = properties)
            
        if dfActive.filter(col('ruleCategory').isNull()).count() == 0:
            logger.info("ruleCategory has no null entries", extra = properties)
            if dfActive.filter(col('ruleCategory').isin(ruleCategoryList)):
                logger.info("ruleCatergory is within defined ruleCategory list", extra = properties)
            else:
                logger.exception("ruleCatergory is not within defined ruleCategory list", extra = properties)
        else:
            logger.exception("ruleCategory has null entries", extra = properties)
    else:
        logger.exception("No active entries with 'y' value in rulemaster file", extra = properties)
                   
except Exception as e:
    logger.exception("Invalid entries found in ruleMaster file", extra = properties)
    raise e

#### Curation Table validation

In [0]:
try:
    logger.info(f"Checking Curation table exists for {sourceObjectName} at the curated", extra = properties)
    
    # deltaTableCheck defined in util notebook to check if delta table exist for sourceObjectName
    tableValidStatus = deltaTableCheck(deltaPath)
    if tableValidStatus == True: 
        logger.info(f"DQ framework started for {sourceObjectName} at the curation", extra = properties)
        logger.info(f"Searching {sourceObjectName} entries in rulemaster and ruleobjectmapping for {sourceObjectName}")
        
        # filtering ruleObjectMapping based on sourceObjectName
        dfFilter = dfRuleObjectMapping.filter((lower(dfRuleObjectMapping.sourceObjectName) == sourceObjectName) & (lower(dfRuleObjectMapping.sourceSystem) == sourceSystem))
        dfJoined = dfRuleMaster.join(dfFilter, "ruleID", "inner")
        # Filter the joined DataFrame based on the conditions isActive = Y and isMappingActive = Y
        dfFilterActive = dfJoined.filter(lower(dfJoined.isActive) == 'y') \
                         .filter(lower(dfJoined.mappingIsActive) == 'y')
        
        # Select the required columns from param notebook
        dfSelected = dfFilterActive.select(*filterColumnList)
        
        # to avoid any duplicate entries and choosing mappingID with smaller mappingID 
        w = Window.partitionBy("sourceObjectName","sourceSystem","ruleID")
        dfSelected = dfSelected.withColumn("criticalityOrder", when((col("criticality") == "High"), lit(1)) \
                                                      .when((col("criticality") == "Medium"), lit(2)) \
                                                      .when((col("criticality") == "Low"), lit(3)))
        
        dfSelected = dfSelected.withColumn("ranks", dense_rank().over(w.orderBy(col("criticalityOrder").asc(),col("mappingID").desc())))
        dfSelected = dfSelected.filter(col("ranks") == 1)
        dfSelected = dfSelected.drop("ranks", "criticalityOrder")
        
        # Select distinct values
        dfSelected = dfSelected.orderBy("ruleID")
        
        logger.info(f"Entries for {sourceObjectName} are joined successfully for active flags from rulemaster and ruleobjectmapping.", extra = properties)
        if dfSelected.count() != 0:
            # to removed selected dataframe
            logger.info(f"Entries for {sourceObjectName} has count greater than zero, rules are present.", extra = properties)
        else:
            logger.exception(f"Entries for {sourceObjectName} has zero count, suggesting there is no rules present", extra = properties)
            dbutils.notebook.exit(f"Notebook exited since there is no value for {sourceObjectName} to process")
    else:
        logger.exception(f"Curation table not found for {sourceObjectName} and code has been terminated", extra = properties)
        raise Exception(f"Curation table not found for {sourceObjectName} and code has been terminated")
        
except Exception as e:
    logger.exception(f"Curation table validation function failed for {sourceObjectName} and code has been terminated", extra = properties)
    raise e

### masterRule Function call executed

In [0]:
try:
    logger.info(f"DQ checks initiated for {sourceObjectName} based on provided configuration", extra = properties)
    for i in dfSelected.collect():
        masterRules(auditAdbPipelineID, i.sourceSystem, i.sourceObjectName, i.targetObjectName, i.columnName, i.ruleName, i.ruleID, i.mappingID, i.criticality ,i.sourceRuleQuery, i.targetRuleQuery, dateColumnName, primaryKey, lastRunDate, currentRunDate)
    logger.info(f"DQ checks successfully completed for {sourceObjectName} based on provided configuration", extra = properties)
except Exception as e:
    logger.exception(f"DQ checks failed for {sourceObjectName} with error as {e}", extra = properties)
    raise e

###  End of dqExecution

In [0]:
dbutils.notebook.exit("DQ run was successful")